# Fine-tune YOLOv8n on CCT20 subset (Colab GPU)

Clones the repo, rebuilds the small data subset from scratch (data is never committed to git), fine-tunes YOLOv8n, and evaluates per-class precision/recall/mAP on the unseen-location test set.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
!git clone https://github.com/ada-th25/camera-trap-site-robustness.git
%cd camera-trap-site-robustness

Install only the lightweight packages individually. We deliberately do **not** run `pip install -r requirements.txt` here — that file was frozen from a local Mac (CPU/MPS) environment and includes a `torch` build that would overwrite Colab's preinstalled CUDA-matched PyTorch, silently losing GPU acceleration.

In [ ]:
!pip install -q ultralytics pandas pyyaml requests tqdm

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Rebuild the data pipeline (same scripts as local dev)

In [ ]:
!mkdir -p data/raw/annotations/extracted && \
curl -L -o data/raw/annotations/eccv_18_annotations.tar.gz https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz && \
tar -xzf data/raw/annotations/eccv_18_annotations.tar.gz -C data/raw/annotations/extracted

from pathlib import Path
expected = Path('data/raw/annotations/extracted/eccv_18_annotation_files/train_annotations.json')
assert expected.exists(), f'{expected} missing - annotation download/extract failed, check the cell output above'
print('annotations ready')

In [ ]:
!PYTHONPATH=. python scripts/download_subset.py

In [ ]:
!PYTHONPATH=. python scripts/build_manifest.py
!PYTHONPATH=. python scripts/convert_to_yolo.py

from pathlib import Path
assert Path('data/yolo/dataset.yaml').exists(), 'dataset.yaml missing - check the cell output above for errors'
print('dataset.yaml ready')

## Fine-tune

`patience=20`: stop early if val mAP hasn't improved in 20 epochs, so we don't burn free-tier GPU time on a plateaued run.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='data/yolo/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project='runs/finetune',
    name='cct20_yolov8n',
)

## Evaluate on the unseen-location test set (location 125)

This gives real per-class precision/recall/mAP50/mAP50-95, computed by Ultralytics' built-in validator — the same metric machinery, now with matching classes.

In [ ]:
best = YOLO('runs/finetune/cct20_yolov8n/weights/best.pt')
metrics = best.val(data='data/yolo/dataset.yaml', split='test')
print(metrics.box.map50, metrics.box.map)

## Download the fine-tuned weights back to your machine

In [ ]:
from google.colab import files
files.download('runs/finetune/cct20_yolov8n/weights/best.pt')